# Stage 0 smoke test on Kaggle

Settings before running: **Accelerator = GPU T4 x2**, **Internet = on**.
If the GitHub repo is private, add a Kaggle secret named `GITHUB_TOKEN` (Add-ons → Secrets) holding a fine-grained token with read access to this repo.

Checks: GPU info, vLLM install, each tier (1.5B, 3B, 7B) answers "What is the capital of France?" correctly and deterministically. Takes about 20–30 minutes, mostly model downloads.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv

In [ ]:
import os, subprocess
REPO = 'himanshugithub360/Joint-Controller-RAG-'
token = ''
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception:
    print('No GITHUB_TOKEN secret; cloning without it (works only if the repo is public).')
url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
if not os.path.exists('/kaggle/working/repo'):
    subprocess.run(['git', 'clone', '--depth', '1', url, '/kaggle/working/repo'], check=True)
%cd /kaggle/working/repo

In [ ]:
# Kaggle's image runs Python 3.13, but vLLM 0.6.6 (the last line that supports T4 GPUs)
# depends on xformers wheels that exist only up to Python 3.12. So vLLM goes into its own
# Python 3.12 environment, made with uv, in /tmp (keeps /kaggle/working small).
!pip install -q uv
!uv venv -q --python 3.12 /tmp/vllm-env
!uv pip install -q --python /tmp/vllm-env/bin/python -r envs/kaggle-requirements.txt
!/tmp/vllm-env/bin/python -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
# One process per tier so GPU memory is fully released between models.
PY = '/tmp/vllm-env/bin/python'
!rm -f smoke_results.jsonl
!{PY} kaggle/smoke_tier.py --model Qwen/Qwen2.5-1.5B-Instruct
!{PY} kaggle/smoke_tier.py --model Qwen/Qwen2.5-3B-Instruct
!{PY} kaggle/smoke_tier.py --model Qwen/Qwen2.5-7B-Instruct --tp 2

In [ ]:
!{PY} scripts/s00_check_env.py --label kaggle
import pandas as pd
pd.read_json('smoke_results.jsonl', lines=True) if os.path.exists('smoke_results.jsonl') else print('No results: read the errors printed by the cell above.')

Download `reports/stage0_env_kaggle.md` and `smoke_results.jsonl` from the output panel and commit them to `reports/` in the repo.